# Jaipur SAR-Based Urban Expansion Analysis
## Phase-wise Jupyter Notebook

**Study area:** Jaipur Municipal Corporation, Rajasthan, India  
**Sensor:** Sentinel-1A IW GRDH, dual-polarization VV/VH  
**Acquisition years:** 2016, 2020 and 2025

**Important interpretation note:** This workflow analyses multi-temporal SAR backscatter and maps surface-change candidates. It does not produce a validated built-up classification or independently prove urban expansion. Current OpenStreetMap (OSM) data are a contemporary reference, not historical ground truth.

This notebook organises the existing project scripts into ten research phases. It runs the scripts from the project `scripts/` folder rather than duplicating their implementation.

## Workflow and script mapping

| Phase | Subject | Scripts |
|---|---|---|
| 1 | Sentinel-1 validation and VV/VH extraction | 01–03 |
| 2 | Raster alignment and initial quality | 04–05 |
| 3 | Diagnostics, metadata, NoData and common footprint | 06–14 |
| 4 | Temporal backscatter analysis | 15 |
| 5 | SAR feature engineering | 16–17, 31–32 |
| 6 | Municipal boundary and OSM preparation | 18–28 |
| 7 | Municipal SAR coverage and pixel audit | 29–30 |
| 8 | SAR change-candidate detection | 33 |
| 9 | Spatial clustering and OSM comparison | 34–35 |
| 10 | Final maps, statistics and temporal visualisation | 36–40 |

Run the notebook from the project root. Review each phase's outputs before proceeding.

## 0. Environment and project setup

Expected project structure: `data/`, `reports/`, `scripts/`, `qgis/`, and the root README.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np

# Run this notebook from the project root, or set the absolute path here.
PROJECT_DIR = Path.cwd()

# Example:
# PROJECT_DIR = Path(r"C:\Users\ssat-\Documents\RemoteSensing_Projects\Project_02_Jaipur_SAR_Urban_Expansion")

SCRIPTS_DIR = PROJECT_DIR / "scripts"
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_DIR / "reports"

assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
assert SCRIPTS_DIR.exists(), f"Scripts directory not found: {SCRIPTS_DIR}"

print("Project:", PROJECT_DIR)
print("Python:", sys.version)
print("Scripts:", len(list(SCRIPTS_DIR.glob("*.py"))))

### Execution helper

The helper stops when a script fails. It does not silently skip errors. Some processing stages may take several minutes.

In [ ]:
def run_scripts(script_names):
    """Run existing project scripts sequentially from the project root."""
    for script_name in script_names:
        script_path = SCRIPTS_DIR / script_name
        if not script_path.exists():
            raise FileNotFoundError(f"Missing script: {script_path}")

        print("\n" + "=" * 75)
        print("RUNNING:", script_name)
        print("=" * 75)

        result = subprocess.run(
            [sys.executable, str(script_path)],
            cwd=PROJECT_DIR,
            text=True,
            capture_output=True
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print("STDERR:\n", result.stderr)
        if result.returncode != 0:
            raise RuntimeError(
                f"{script_name} failed with exit code {result.returncode}"
            )

def list_outputs(folder, pattern="*"):
    folder = Path(folder)
    if not folder.exists():
        print("Not created yet:", folder)
        return
    for item in sorted(folder.glob(pattern)):
        if item.is_file():
            print(f"{item.relative_to(PROJECT_DIR)} | {item.stat().st_size / 1024**2:.2f} MB")

# Phase 1 — Sentinel-1 data validation and VV/VH extraction

**Objective:** Validate the six terrain-corrected VV/VH inputs for 2016, 2020 and 2025, extract single-band rasters and validate the extracted outputs.

**Scripts:** `01_validate_sar_rasters.py`, `02_extract_vv_vh_bands.py`, `03_validate_extracted_bands.py`.

**Inputs:** `data/processed/` terrain-corrected rasters.

In [ ]:
run_scripts([
    "01_validate_sar_rasters.py",
    "02_extract_vv_vh_bands.py",
    "03_validate_extracted_bands.py"
])
list_outputs(PROCESSED_DIR, "*singleband.tif")

# Phase 2 — Spatial alignment and initial raster quality

**Objective:** Align the six single-band rasters to a common grid and assess initial raster quality.

**Recorded target grid:** EPSG:32643, 10 m resolution, 3993 × 3349 pixels.

**Scripts:** `04_align_sar_rasters.py`, `05_assess_raster_quality.py`.

Check CRS, dimensions, pixel size and transform before pixel-wise comparisons.

In [ ]:
run_scripts([
    "04_align_sar_rasters.py",
    "05_assess_raster_quality.py"
])
ALIGNED_DIR = PROCESSED_DIR / "aligned_utm43n"
list_outputs(ALIGNED_DIR, "*.tif")

# Phase 3 — Raster diagnostics, metadata, NoData and common footprint

**Objective:** Examine backscatter distributions and extreme pixels, verify available metadata and processing records, audit NoData footprints, and calculate the common valid-data footprint.

**Scripts:** `06` through `14`.

Retain these diagnostics as supporting evidence, even when they are not used in the final figures.

In [ ]:
run_scripts([
    "06_backscatter_distribution_analysis.py",
    "07_investigate_extreme_pixels.py",
    "08_inspect_extreme_vv_locations.py",
    "09_visualize_extreme_locations.py",
    "10_verify_sentinel1_metadata.py",
    "11_check_processing_records.py",
    "12_verify_nodata_and_raster_footprints.py",
    "13_visualize_nodata_footprints.py",
    "14_calculate_common_valid_footprint.py"
])
list_outputs(REPORTS_DIR / "quality_analysis")

# Phase 4 — Temporal backscatter analysis

**Objective:** Compare VV and VH backscatter across the three acquisition years.

**Script:** `15_temporal_backscatter_comparison.py`.

Positive backscatter shifts describe changes in radar response. They are not direct measurements of built-up-area growth. Surface moisture, vegetation, roughness and acquisition conditions can also affect backscatter.

In [ ]:
run_scripts(["15_temporal_backscatter_comparison.py"])
list_outputs(REPORTS_DIR / "temporal_backscatter_analysis")

# Phase 5 — SAR feature engineering

**Objective:** Derive dual-polarization features and multi-year feature layers from aligned VV/VH rasters.

**Scripts:** `16_sar_feature_engineering.py`, `17_visualize_sar_features.py`, `31_extract_sar_features.py`, `32_visualize_sar_features.py`.

The project includes dual-polarization total-backscatter and VV/VH-ratio features. Confirm definitions and units from the scripts and raster metadata before interpreting them.

In [ ]:
run_scripts([
    "16_sar_feature_engineering.py",
    "17_visualize_sar_features.py",
    "31_extract_sar_features.py",
    "32_visualize_sar_features.py"
])
FEATURE_DIR = PROCESSED_DIR / "sar_features"
list_outputs(FEATURE_DIR, "*.tif")

# Phase 6 — Jaipur boundary and OSM reference preparation

**Objective:** Inspect and visualise the municipal boundary, create the OSM extraction extent, audit and clip the OSM reference dataset, and assess tags and spatial coverage.

**Scripts:** `18` through `28`.

**Limitation:** OSM represents mapped features available at extraction time. It is not historical ground truth for 2016 or 2020. Document boundary provenance and OSM attribution in the project README.

In [ ]:
run_scripts([
    "18_inspect_jaipur_municipal_boundary.py",
    "19_visualize_jaipur_boundary.py",
    "20_create_osm_download_extent.py",
    "21_audit_osm_reference_data.py",
    "22_clip_osm_reference_data.py",
    "23_investigate_large_landuse.py",
    "24_validate_clipped_osm.py",
    "25_analyze_osm_classification_tags.py",
    "26_calculate_osm_landuse_area.py",
    "27_calculate_osm_municipal_coverage.py",
    "28_visualize_osm_municipal_coverage.py"
])
list_outputs(REPORTS_DIR / "osm_quality_analysis")

# Phase 7 — Municipal SAR coverage and pixel-value audit

**Objective:** Assess the valid SAR footprint within the municipal boundary and audit pixel values and feature-extraction quality.

**Scripts:** `29_assess_sar_municipal_coverage.py`, `30_audit_sar_pixel_values.py`.

A valid aligned raster mask over the municipal polygon does not, by itself, prove that the original satellite observation was complete or free from all acquisition and processing limitations.

In [ ]:
run_scripts([
    "29_assess_sar_municipal_coverage.py",
    "30_audit_sar_pixel_values.py"
])
list_outputs(REPORTS_DIR / "sar_coverage_analysis")

# Phase 8 — SAR change-candidate detection

**Objective:** Calculate temporal VV/VH change layers and identify candidate pixels using the project's joint backscatter-change thresholds.

**Script:** `33_detect_sar_change_candidates.py`.

Recorded 2016–2025 joint p95 thresholds were approximately VV = 5.866 dB and VH = 3.916 dB. These identify strong joint backscatter increases under the selected method; they are not automatically built-up pixels.

In [ ]:
run_scripts(["33_detect_sar_change_candidates.py"])
CHANGE_DIR = PROCESSED_DIR / "change_detection"
list_outputs(CHANGE_DIR, "*.tif")

# Phase 9 — Spatial clustering and OSM comparison

**Objective:** Group spatially connected change candidates and compare their locations with current OSM building references.

**Scripts:** `34_identify_spatial_change_clusters.py`, `35_compare_clusters_with_osm.py`.

Recorded result: 196 retained joint high-change clusters larger than 0.5 ha, with a combined mapped area of approximately 2.3042 km². OSM overlap is exploratory and is not a historical accuracy assessment.

In [ ]:
run_scripts([
    "34_identify_spatial_change_clusters.py",
    "35_compare_clusters_with_osm.py"
])
list_outputs(CHANGE_DIR / "spatial_clusters")
list_outputs(REPORTS_DIR / "change_detection")

# Phase 10 — Final maps, statistics and temporal visualisation

**Objective:** Generate consolidated maps, final result tables, statistical summaries and multi-period temporal visualisations.

**Scripts:** `36` through `40`.

Expected report locations include `change_detection/`, `final_results/`, `final_statistics_v2/` and `final_temporal_change_analysis/`.

In [ ]:
run_scripts([
    "36_generate_final_project_maps.py",
    "37_compile_final_project_results.py",
    "38_generate_multiyear_sar_maps.py",
    "39_generate_final_sar_statistics.py",
    "40_generate_jaipur_temporal_change_visualization.py"
])
for folder in [
    REPORTS_DIR / "final_results",
    REPORTS_DIR / "final_statistics_v2",
    REPORTS_DIR / "final_temporal_change_analysis"
]:
    print("\nOUTPUTS:", folder.relative_to(PROJECT_DIR))
    list_outputs(folder)

# Final interpretation and limitations

## Supported by this analysis
- Multi-year Sentinel-1 VV/VH backscatter comparison for 2016, 2020 and 2025.
- Spatial mapping of SAR backscatter-change candidates.
- Statistical description of backscatter distributions and temporal differences.
- Identification of spatially connected high-change clusters.
- Exploratory comparison of candidate clusters with current OSM building footprints.

## Not established by this analysis
- A validated built-up/non-built-up classification.
- Historical building footprints or historical urban extent.
- A validated urban expansion rate or built-up-area growth figure.
- Classification accuracy based on current OSM overlap.

## Recommended future validation
1. Obtain historical optical imagery or authoritative built-up reference data for comparable dates.
2. Develop and validate a built-up classification using independent reference samples.
3. Use a documented sampling design and confusion matrix.
4. Only then estimate built-up area change and urban expansion rates.

## Reproducibility notes
- Retain original Sentinel-1 `.SAFE` products as raw inputs; do not upload them directly to GitHub.
- Keep large GeoTIFF and GeoPackage files outside GitHub and document their provenance and retrieval/regeneration instructions.
- SNAP intermediate `.dim` products were removed in the earlier workflow. Do not claim unavailable processing records were independently verified.
- Provide appropriate OSM attribution and comply with the Open Database License (ODbL).

# Project completion checklist

- [ ] Six Sentinel-1 input rasters validated.
- [ ] VV/VH single-band rasters generated and checked.
- [ ] Six rasters aligned to the common projected grid.
- [ ] NoData and common-footprint checks reviewed.
- [ ] Temporal backscatter statistics and plots reviewed.
- [ ] SAR features generated and inspected.
- [ ] Municipal boundary provenance and geometry checks documented.
- [ ] OSM extraction, clipping and quality checks reviewed.
- [ ] Municipal SAR coverage and pixel audits reviewed.
- [ ] Change-candidate thresholds documented.
- [ ] Spatial clusters and OSM comparison reviewed.
- [ ] Final maps and statistical outputs checked.
- [ ] Limitations and future validation requirements included in the final report.